# 00 Ambient RNA Correction (scAR)

This notebook performs ambient RNA correction on RAW 10x matrices using scAR and exports per-sample corrected outputs for downstream QC, doublet detection, and integration.

## Outline
1. Discover RAW input files.
2. Run scAR per sample and save corrected sparse counts + metadata.
3. Export corrected `*.h5ad` files to `Analysis/scAR/h5ad/`.
4. Write run manifests for reproducible handoff to `01_Ingest.ipynb`.

## Output Naming Policy
- This notebook uses `NOTEBOOK_TAG = "nb00"` and central path constants for exported manifests/tables.
- Per-sample corrected `.h5ad` names remain `*_scar_corrected.h5ad` for compatibility with notebook 01 ingestion.
- Intermediate scAR matrices and metadata files include the notebook tag (for traceability).

In [ ]:
# Step 1: Project setup and RAW matrix discovery
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch

# -------------------------------------------------------------------
# Project paths and notebook-aware naming
# -------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "Notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "Notebooks").resolve()
ANALYSIS_DIR = NOTEBOOK_DIR.parent.resolve()
PROJECT_DIR = ANALYSIS_DIR.parent.resolve()
PER_SAMPLE_OUTS_DIR = PROJECT_DIR / "per_sample_outs"

NOTEBOOK_TAG = "nb00"

# Permanent ambient-correction outputs
SCAR_DIR = ANALYSIS_DIR / "scAR"
SCAR_COUNT_DIR = SCAR_DIR / "counts"
SCAR_H5AD_DIR = SCAR_DIR / "h5ad"

RAW_MANIFEST_PATH = SCAR_DIR / f"raw_samples_manifest_{NOTEBOOK_TAG}.csv"
SCAR_RESULTS_PATH = SCAR_DIR / f"scar_results_{NOTEBOOK_TAG}.csv"
SCAR_SUCCESS_PATH = SCAR_DIR / f"scar_success_manifest_{NOTEBOOK_TAG}.csv"
SCAR_H5AD_MANIFEST_PATH = SCAR_DIR / f"scar_h5ad_manifest_{NOTEBOOK_TAG}.csv"

for out_dir in [SCAR_DIR, SCAR_COUNT_DIR, SCAR_H5AD_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# QC and scAR configuration
# -------------------------------------------------------------------
MIN_GENES_THRESHOLD = 50
MIN_COUNTS_THRESHOLD = 100
MIN_CELLS_THRESHOLD = 3

SCAR_FEATURE_TYPE = "mRNA"
SCAR_PROB = 0.995
SCAR_SETUP_ITER = 3
SCAR_EPOCHS = 150
SCAR_BATCH_SIZE = 64
SCAR_DEVICE = "auto"

USE_GPU = torch.cuda.is_available()
print(f"NOTEBOOK_TAG={NOTEBOOK_TAG}")
print(f"USE_GPU={USE_GPU}")
print(f"torch={torch.__version__}")
print(f"SCAR_DIR={SCAR_DIR}")
print(f"RAW_MANIFEST_PATH={RAW_MANIFEST_PATH}")

# -------------------------------------------------------------------
# Discover RAW 10x H5 inputs
# -------------------------------------------------------------------
raw_rows = []
for sample_dir in sorted(PER_SAMPLE_OUTS_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue

    count_dir = sample_dir / "count"
    raw_candidates = [
        count_dir / "sample_raw_feature_bc_matrix.h5",
        count_dir / "raw_feature_bc_matrix.h5",
    ]
    raw_h5 = next((p for p in raw_candidates if p.is_file()), None)

    if raw_h5 is not None:
        raw_rows.append({"Sample": sample_dir.name, "raw_h5": str(raw_h5)})

sample_df = pd.DataFrame(raw_rows).sort_values("Sample").reset_index(drop=True)
if sample_df.empty:
    raise ValueError("No RAW h5 files found under per_sample_outs/*/count.")

print("RAW samples discovered:")
display(sample_df)

sample_df.to_csv(RAW_MANIFEST_PATH, index=False)
print(RAW_MANIFEST_PATH)


In [ ]:
# Step 2: Install/verify scAR and define sample-level runner
import importlib
import importlib.metadata as im
import subprocess
import sys
import time
from typing import Any, Dict

import scipy.sparse as sps


def ensure_scar_available() -> Any:
    """Ensure Novartis scAR is importable in the active environment."""
    try:
        dist = im.distribution("scar")
        summary = (dist.metadata.get("Summary", "") or "").lower()
        metadata_text = str(dist.metadata).lower()
        if "serverless" in summary or "grycap" in metadata_text:
            print("Removing unrelated scar package...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "uninstall", "-y", "scar"]
            )
    except Exception:
        pass

    need_install = True
    try:
        import scar  # noqa: F401

        if hasattr(scar, "setup_anndata") and hasattr(scar, "model"):
            need_install = False
    except Exception:
        need_install = True

    if need_install:
        print("Installing scAR (ambient RNA) from GitHub...")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "git+https://github.com/Novartis/scar.git",
            ]
        )

    import scar

    importlib.reload(scar)
    print("scar module path:", scar.__file__)
    print("scar version:", getattr(scar, "__version__", "unknown"))
    return scar


scar = ensure_scar_available()


def run_scar_raw(
    input_h5: str,
    sample: str,
    min_genes: int = MIN_GENES_THRESHOLD,
    min_counts: int = MIN_COUNTS_THRESHOLD,
    min_cells: int = MIN_CELLS_THRESHOLD,
    prob: float = SCAR_PROB,
    setup_iter: int = SCAR_SETUP_ITER,
    epochs: int = SCAR_EPOCHS,
    batch_size: int = SCAR_BATCH_SIZE,
    device: str = SCAR_DEVICE,
) -> Dict[str, Any]:
    """Run scAR on one sample and save corrected sparse counts + metadata."""
    t0 = time.time()
    print(f"[{sample}] scAR start")

    raw_adata = sc.read_10x_h5(input_h5, genome=None, gex_only=True)
    raw_adata.var_names_make_unique()

    flt_adata = raw_adata.copy()
    sc.pp.filter_cells(flt_adata, min_genes=min_genes)
    sc.pp.filter_cells(flt_adata, min_counts=min_counts)
    sc.pp.filter_genes(flt_adata, min_cells=min_cells)

    if flt_adata.n_obs < 50 or flt_adata.n_vars < 200:
        raise ValueError(
            f"{sample}: too few cells/genes after prefilter {flt_adata.shape}"
        )

    scar.setup_anndata(
        adata=flt_adata,
        raw_adata=raw_adata,
        feature_type=SCAR_FEATURE_TYPE,
        prob=prob,
        iterations=setup_iter,
        kneeplot=False,
        verbose=True,
    )

    model = scar.model(
        flt_adata,
        feature_type=SCAR_FEATURE_TYPE,
        device=device,
        verbose=True,
    )
    model.train(
        epochs=epochs,
        batch_size=batch_size,
        save_model=False,
        verbose=True,
    )
    model.inference()

    native = model.native_counts

    out_counts_npz = SCAR_COUNT_DIR / f"{sample}_scar_native_counts_{NOTEBOOK_TAG}.npz"
    out_cells = SCAR_DIR / f"{sample}_scar_cells_{NOTEBOOK_TAG}.csv"
    out_genes = SCAR_DIR / f"{sample}_scar_genes_{NOTEBOOK_TAG}.csv"

    if sps.issparse(native):
        sps.save_npz(out_counts_npz, native.tocsr())
    else:
        arr = np.asarray(native)
        expected_shape = (flt_adata.n_obs, flt_adata.n_vars)
        transposed_shape = (flt_adata.n_vars, flt_adata.n_obs)

        if arr.shape == transposed_shape:
            arr = arr.T
        elif arr.shape != expected_shape:
            raise ValueError(
                f"{sample}: unexpected native_counts shape {arr.shape}; "
                f"expected {expected_shape} or {transposed_shape}"
            )

        sps.save_npz(out_counts_npz, sps.csr_matrix(arr))

    flt_adata.obs.to_csv(out_cells)
    flt_adata.var.to_csv(out_genes)

    elapsed_sec = time.time() - t0
    print(
        f"[{sample}] scAR done in {elapsed_sec:.1f}s | "
        f"cells={flt_adata.n_obs}, genes={flt_adata.n_vars}"
    )

    return {
        "Sample": sample,
        "raw_h5": input_h5,
        "status": "ok",
        "seconds": elapsed_sec,
        "n_cells": int(flt_adata.n_obs),
        "n_genes": int(flt_adata.n_vars),
        "scar_counts_npz": str(out_counts_npz),
        "cells_meta": str(out_cells),
        "genes_meta": str(out_genes),
        "error": "",
    }


print("run_scar_raw defined - ready for Cell 3")


In [ ]:
# Step 3: Run scAR across all discovered RAW samples
scar_results = []

for idx, row in sample_df.iterrows():
    sample = row["Sample"]
    raw_h5 = row["raw_h5"]
    print(f"\nRunning scAR {idx + 1}/{len(sample_df)}: {sample}")

    try:
        result = run_scar_raw(raw_h5, sample, epochs=SCAR_EPOCHS)
    except Exception as exc:
        result = {
            "Sample": sample,
            "raw_h5": raw_h5,
            "status": "failed",
            "seconds": np.nan,
            "n_cells": np.nan,
            "n_genes": np.nan,
            "scar_counts_npz": "",
            "cells_meta": "",
            "genes_meta": "",
            "error": f"{type(exc).__name__}: {exc}",
        }

    scar_results.append(result)

scar_df = pd.DataFrame(scar_results)
display(scar_df)

scar_df.to_csv(SCAR_RESULTS_PATH, index=False)
scar_df.loc[scar_df["status"] == "ok"].to_csv(SCAR_SUCCESS_PATH, index=False)

print(SCAR_RESULTS_PATH)
print(SCAR_SUCCESS_PATH)
print(f"Success: {(scar_df['status'] == 'ok').sum()} / {len(scar_df)}")


In [ ]:
# Step 4: Build per-sample corrected .h5ad from scAR outputs
from pathlib import Path

import anndata as ad
import scipy.sparse as sps

SCAR_H5AD_DIR.mkdir(parents=True, exist_ok=True)

h5ad_rows = []

# Prefer the run manifest from Cell 3
if SCAR_RESULTS_PATH.exists():
    run_df = pd.read_csv(SCAR_RESULTS_PATH)
    run_df = run_df.loc[run_df["status"] == "ok"].copy()
else:
    # Fallback: infer expected scAR output paths from discovered samples
    run_df = sample_df.copy()
    run_df["scar_counts_npz"] = run_df["Sample"].map(
        lambda s: str(SCAR_COUNT_DIR / f"{s}_scar_native_counts_{NOTEBOOK_TAG}.npz")
    )
    run_df["cells_meta"] = run_df["Sample"].map(
        lambda s: str(SCAR_DIR / f"{s}_scar_cells_{NOTEBOOK_TAG}.csv")
    )
    run_df["genes_meta"] = run_df["Sample"].map(
        lambda s: str(SCAR_DIR / f"{s}_scar_genes_{NOTEBOOK_TAG}.csv")
    )

for _, row in run_df.iterrows():
    sample = row["Sample"]
    npz_path = Path(str(row["scar_counts_npz"]))
    cells_path = Path(str(row["cells_meta"]))
    genes_path = Path(str(row["genes_meta"]))

    if not (npz_path.exists() and cells_path.exists() and genes_path.exists()):
        h5ad_rows.append(
            {
                "Sample": sample,
                "status": "missing_inputs",
                "h5ad": "",
                "error": (
                    "Missing one or more files: "
                    f"{npz_path.name}, {cells_path.name}, {genes_path.name}"
                ),
            }
        )
        continue

    try:
        X = sps.load_npz(npz_path).tocsr()
        obs = pd.read_csv(cells_path, index_col=0)
        var = pd.read_csv(genes_path, index_col=0)

        if X.shape != (obs.shape[0], var.shape[0]):
            raise ValueError(
                f"Shape mismatch: X={X.shape}, obs={obs.shape}, var={var.shape}"
            )

        adata_corr = ad.AnnData(X=X, obs=obs, var=var)
        adata_corr.obs_names = obs.index.astype(str)
        adata_corr.var_names = var.index.astype(str)
        adata_corr.var_names_make_unique()

        adata_corr.uns["ambient_correction_method"] = "scAR"
        adata_corr.uns["ambient_correction_input"] = str(npz_path)

        # Keep corrected h5ad filename compatible with notebook 01 ingestion glob.
        out_h5ad = SCAR_H5AD_DIR / f"{sample}_scar_corrected.h5ad"
        adata_corr.write_h5ad(out_h5ad, compression="gzip")

        h5ad_rows.append(
            {
                "Sample": sample,
                "status": "ok",
                "h5ad": str(out_h5ad),
                "n_cells": int(adata_corr.n_obs),
                "n_genes": int(adata_corr.n_vars),
                "error": "",
            }
        )
    except Exception as exc:
        h5ad_rows.append(
            {
                "Sample": sample,
                "status": "failed",
                "h5ad": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

h5ad_df = pd.DataFrame(h5ad_rows)
display(h5ad_df)

h5ad_df.to_csv(SCAR_H5AD_MANIFEST_PATH, index=False)
print(SCAR_H5AD_MANIFEST_PATH)

if h5ad_df.empty or "status" not in h5ad_df.columns:
    raise RuntimeError(
        "No corrected h5ad rows were produced. Check notebook 00 Step 3 logs and the scAR input manifests."
    )

print(f"h5ad built: {(h5ad_df['status'] == 'ok').sum()} / {len(h5ad_df)}")
